# Ticha HTR — using the model

How to read a manuscript line with [`maclenn77/ticha-trocr-base`](https://huggingface.co/maclenn77/ticha-trocr-base), TrOCR base fine-tuned on Colonial Zapotec and Spanish manuscripts from [Ticha](https://ticha.haverford.edu/).

- **Input:** the image of **one text line**, cropped from a page (the model does not find lines on a page).
- **Output:** the line as text, as a diplomatic transcription: what is written, abbreviations not expanded.

The repository holds two models. The one at the root reads the line (left to right). The one in `r2l/` reads candidate lines backwards and is only used to rerank: decoding with 10 beams and reranking them with it is the best system (test CER 13.6%, against 14.1% for 10 beams alone).

Runs on a GPU, an Apple Silicon GPU (MPS) or a CPU (slower: a few seconds per line). No training is needed: the models and the dataset are downloaded from the Hugging Face Hub.

**Requirements:** from the repository root, `pip install -e '.[htr]' datasets jiwer` (the notebook imports `ticha` from `src/`, so an installed package is not required), or the **Python (ticha)** environment of [`environment.yml`](../../environment.yml), which has everything this notebook needs (see [`ENVIRONMENT.md`](../../ENVIRONMENT.md)).

In [ ]:
import sys
from pathlib import Path

from datasets import load_dataset
from IPython.display import display
from jiwer import cer

# The kernel may start in the repo root (VS Code, the VM's Jupyter service) or in notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'ticha').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from ticha.core.htr import RERANK_WEIGHT, LineReader, best_candidate, clean_text

reader = LineReader.from_pretrained('maclenn77/ticha-trocr-base')  # rerank=False skips the r2l model
print('device:', reader.device)

## 1. Some test lines

The test split of [`maclenn77/ticha-htr-lines`](https://huggingface.co/datasets/maclenn77/ticha-htr-lines) (version 1.1), which the model never saw.

The transcriptions in the dataset still have the transcribers' notes: `[...]` and `(...)` for expansions, glosses and damage. `clean_text` (in `ticha.core.htr`) is the rule used in training and evaluation: it deletes expansions (`p[es]os` becomes `pos`) and drops lines with damage or illegible text, so the reference matches what is on the page.

In [ ]:
test = load_dataset('maclenn77/ticha-htr-lines', 'original', split='test', revision='v1.1')
examples, seen = [], set()
for row in test:  # the first clean line from line 5 on, in each of 8 documents
    reference = clean_text(row['text'])
    if reference and row['doc_id'] not in seen and row['line'] >= 5:
        examples.append({**row, 'reference': reference})
        seen.add(row['doc_id'])
    if len(examples) == 8:
        break
print(len(examples), 'lines from', sorted(seen))

## 2. One line, step by step

The processor resizes the line image to 384 × 384 pixels and normalizes it. `reader.candidates` runs beam search and returns the 10 best readings, best first, each with its token ids and two scores: the summed log-probability that the left-to-right model gives it, and the one the right-to-left model gives it read backwards.

In [ ]:
example = examples[0]
display(example['image'])
print(f"{example['doc_id']}, page {example['page']}, line {example['line']} ({example['language']})")
print('dataset text:', example['text'])
print('reference:   ', example['reference'])
print('model input: ', tuple(reader.pixel_values([example['image']]).shape))

candidates = reader.candidates([example['image']], beams=10)[0]
for k, c in enumerate(candidates):
    print(f'{k}: {c.text}')

## 3. Reranking with the right-to-left model

The combined score of each candidate is

`((1 − w) · log p_left-to-right + w · log p_right-to-left) / number of tokens`

with `w = 0.3`, the weight chosen on validation for 10 beams (`RERANK_WEIGHT`). The candidate with the best score is the reading.

The rerank lowers the CER on average (0.55 points on the test set), but it does not improve every line: on some it picks a candidate that reads worse than the first beam.

In [ ]:
best = best_candidate(candidates, RERANK_WEIGHT)
for k, c in enumerate(candidates):
    print(f"{'>' if c is best else ' '} {k}: {c.score():7.3f}  (L2R {c.l2r:7.2f}, R2L {c.r2l:7.2f})  {c.text}")
print()
print('10 beams alone:', candidates[0].text, f'| CER {cer(example["reference"], candidates[0].text or " "):.1%}')
print('with rerank:   ', best.text, f'| CER {cer(example["reference"], best.text or " "):.1%}')

## 4. Transcribing lines

`reader.transcribe` takes one image (a PIL image or a path) and returns its text, or a list of images and returns a list of texts. It reranks by default; `rerank=False` keeps the first beam.

In [ ]:
images = [e['image'] for e in examples]
references = [e['reference'] for e in examples]
alone = reader.transcribe(images, rerank=False)
reranked = reader.transcribe(images)
for e, text in zip(examples, reranked):
    display(e['image'])
    print(f"{e['doc_id']} ({e['language']})\n  reference: {e['reference']}\n  model:     {text}\n")
print(f'CER on these {len(references)} lines: 10 beams {cer(references, [t or " " for t in alone]):.1%}, '
      f'with rerank {cer(references, [t or " " for t in reranked]):.1%}')

## 5. Your own images

Crop one text line per image, as tight as the dataset's line images (a little margin around the ink is fine), then:

```python
print(reader.transcribe('my_line.png'))
print(reader.transcribe(['line_1.png', 'line_2.png'], rerank=False))  # faster: one model, no rescoring
```

The model expects a single line in roughly the hands and spellings of 17th to 19th century Oaxacan documents. A whole page, several lines in one image, or printed text will read badly.